# QSAR modelling of human acetylcholinesterase (AChE) inhibitors

**Target:** ChEMBL `CHEMBL220` (human AChE, UniProt P22303) · **Endpoint:** IC₅₀ → pIC₅₀ · **Tasks:** active/inactive classification and pIC₅₀ regression

This notebook walks through the study step by step. All logic lives in the `ache_qsar` package, so the notebook and the command-line pipeline (`python run_pipeline.py`) produce identical results.

> **Why `CHEMBL220`?** ChEMBL target IDs are easy to mix up: `CHEMBL216` is the muscarinic M1 receptor, not AChE. Always verify the target name before modelling.

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import Image, Markdown, display

from ache_qsar import config, plots
from ache_qsar.data import curate, load_or_fetch
from ache_qsar.features import build_features
from ache_qsar.split import scaffold_split

TARGET = config.DEFAULT_TARGET                       # human AChE
INPUT_CSV = os.environ.get("ACHE_INPUT_CSV")         # optional: use a local raw CSV instead of querying ChEMBL
FAST = os.environ.get("ACHE_FAST") == "1"            # tiny grids, for smoke tests only
FIG = ROOT / "results" / "figures"
print("Repository root:", ROOT)

## 1. Data retrieval

All IC₅₀ records for the target are pulled from ChEMBL through its REST API. The raw snapshot is cached in `data/raw/` together with the ChEMBL release number, so the study is reproducible and runs offline afterwards.

In [ ]:
raw, chembl_version = (pd.read_csv(INPUT_CSV), "user-provided file") if INPUT_CSV else load_or_fetch(TARGET, ROOT / "data" / "raw")
print(f"{len(raw):,} raw IC50 records · {chembl_version}")
raw.head(3)

## 2. Curation

Raw ChEMBL data contain censored values (`>`, `<`), mixed units, salts and many repeated measurements of the same compound. Curation keeps **exact IC₅₀ values in nM from biochemical assays**, standardises structures to the neutral parent compound, merges replicate measurements into a **median pIC₅₀** (so the same molecule can never appear in both training and test sets), and removes compounds whose replicates disagree by more than 1 log unit.

`pIC₅₀ = 9 − log₁₀(IC₅₀ [nM])` · **active** = pIC₅₀ ≥ 6 (IC₅₀ ≤ 1 µM) · **inactive** = pIC₅₀ ≤ 5 (IC₅₀ ≥ 10 µM) · compounds in between are used for regression only.

In [ ]:
df, report = curate(raw)
pd.Series(report, name="count").to_frame()

In [ ]:
plots.plot_eda(df, FIG / "01_eda.png", "human AChE")
display(Image(FIG / "01_eda.png"))

## 3. Features and scaffold split

Two complementary representations are compared: **RDKit 2D descriptors** (~210 physicochemical/topological values) and **Morgan count fingerprints** (ECFP4-style, 2048 bits).

The data are split with a **Bemis–Murcko scaffold split**: no ring-system scaffold appears in both the training and test sets, so the test set measures how well the models generalise to *new chemotypes*, which is the realistic use case in drug discovery. Random splits over-estimate performance for QSAR data because close analogues end up on both sides.

In [ ]:
smiles = df["smiles"].tolist()
train_idx, test_idx, scaffolds = scaffold_split(smiles)
shared = len(set(scaffolds[train_idx]) & set(scaffolds[test_idx]))
print(f"train = {len(train_idx)}, test = {len(test_idx)}, scaffolds shared between train and test = {shared}")
X_rdkit = build_features(smiles, "rdkit")
print("RDKit descriptor matrix:", X_rdkit.shape)

## 4. Model training, tuning and evaluation

`run_pipeline.py` (called here through its Python API) trains five model families per task: Random Forest, SVM/SVR, histogram gradient boosting, k-NN and logistic/ridge regression. Each is compared with a trivial baseline and tuned by **scaffold-grouped cross-validation on the training set only**. The final model is selected by CV score, so the test set is touched once, for reporting. Preprocessing (imputation, variance filter, scaling) is part of each scikit-learn `Pipeline`, so nothing leaks across folds.

This step can take a while on the full dataset (grid searches over 5 models × feature sets × 2 tasks).

In [ ]:
from ache_qsar.pipeline import build_parser, run

argv = ["--out", str(ROOT), "--target", TARGET, "--features", "rdkit,ecfp4"]
if INPUT_CSV:
    argv += ["--input-csv", INPUT_CSV]
if FAST:
    argv += ["--fast"]
study = run(build_parser().parse_args(argv))

## 5. Results

In [ ]:
display(Markdown((ROOT / "results" / "RESULTS.md").read_text(encoding="utf-8").replace("figures/", "../results/figures/")))

## 6. Scoring new molecules

The selected models are saved in `models/`. As a sanity check we score four well-known AChE inhibitors. **Caution:** these drugs may well be in the ChEMBL training data, so this demonstrates the interface and is *not* a validation. The `in_applicability_domain` flag warns when a molecule is structurally far from the training set (max Tanimoto < 0.4).

In [ ]:
from ache_qsar.predict import predict

known = {
    "donepezil":    "COc1cc2c(cc1OC)C(=O)C(CC1CCN(Cc3ccccc3)CC1)C2",
    "galantamine":  "CN1CCC23C=CC(O)CC2Oc2c(OC)ccc(C1)c23",
    "rivastigmine": "CCN(C)C(=O)Oc1cccc(C(C)N(C)C)c1",
    "tacrine":      "Nc1c2c(nc3ccccc13)CCCC2",
}
out = predict(list(known.values()), ROOT / "models")
out.insert(0, "name", list(known))
out.drop(columns=["input_smiles", "standardized_smiles"]).round(3)

## 7. Limitations

* **Assay heterogeneity.** ChEMBL IC₅₀ values come from many laboratories, substrates and enzyme sources; median aggregation and the replicate-consistency filter reduce, but cannot remove, this noise, which limits the achievable accuracy.
* **Applicability domain.** Predictions for molecules dissimilar to the training chemistry (see the AD analysis above) are extrapolations.
* **No prospective validation.** Performance is measured on held-out ChEMBL compounds, not on newly synthesised and tested molecules.
* **Next steps:** graph neural networks (e.g. Chemprop), conformer-based 3D descriptors, conformal prediction for calibrated uncertainty, and experimental follow-up of top-ranked compounds.